# 10 8-Lane Realtime Virtual Instrument + PFB/TX Preflight

Stage 7 entry point: 8-lane preview scope/spectrum, DAC tone bank control, PFB channel-window status, and QSFP/CMAC UDP TX preflight route/frame inspection.


In [ ]:
from pathlib import Path
import asyncio
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find overlay/t510_fengine.bit and python/t510_fengine.py')

sys.path.insert(0, str(PROJECT_ROOT))
from python.t510_fengine import T510FEngine

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
EXPECTED_CORE_VERSION = 0x00010005
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#17becf', '#bcbd22']

state = {
    'core': None,
    'live': False,
    'task': None,
    'last_preview': None,
    'last_spectrum': None,
    'last_error': None,
}

download_widget = W.Checkbox(value=True, description='Download bitstream')
frequency_widget = W.FloatSlider(value=5.0, min=0.25, max=30.0, step=0.25, description='Frequency MHz', readout_format='.2f', style={'description_width': '130px'})
dac_rate_widget = W.FloatText(value=245.760, description='DAC rate MHz', style={'description_width': '130px'})
amplitude_widget = W.IntSlider(value=2048, min=0, max=8192, step=128, description='Amplitude', style={'description_width': '130px'})
phase_widget = W.FloatSlider(value=35.0, min=0.0, max=180.0, step=1.0, description='Phase deg/ch', readout_format='.0f', style={'description_width': '130px'})
channels_widget = W.IntSlider(value=4, min=1, max=8, step=1, description='Visible ch', style={'description_width': '130px'})
samples_widget = W.Dropdown(options=[256, 512, 1024], value=512, description='Samples', style={'description_width': '130px'})
interval_widget = W.FloatSlider(value=1.0, min=0.3, max=5.0, step=0.1, description='Refresh s', readout_format='.1f', style={'description_width': '130px'})
timeout_widget = W.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Timeout s', readout_format='.1f', style={'description_width': '130px'})
reference_widget = W.Checkbox(value=True, description='DAC reference')
pfb_chan0_widget = W.IntText(value=0, description='PFB chan0', style={'description_width': '130px'})
pfb_chan_count_widget = W.IntText(value=64, description='PFB chan count', style={'description_width': '130px'})
pfb_time_count_widget = W.IntText(value=4, description='PFB time count', style={'description_width': '130px'})
pfb_taps_widget = W.IntText(value=4, description='PFB taps', style={'description_width': '130px'})
pfb_shift_widget = W.IntText(value=0, description='FFT shift', style={'description_width': '130px'})

load_button = W.Button(description='Load overlay', button_style='primary')
init_button = W.Button(description='Init Stage 7', button_style='success')
apply_button = W.Button(description='Apply DAC')
capture_button = W.Button(description='Capture once')
start_button = W.Button(description='Start live', button_style='success')
stop_button = W.Button(description='Stop live', button_style='warning')
status_button = W.Button(description='Read status')
pfb_button = W.Button(description='Apply PFB')
header_button = W.Button(description='Capture PFB header')
tx_config_button = W.Button(description='Config TX routes')
tx_frame_button = W.Button(description='Capture TX frame')
tx_walk_button = W.Button(description='SPEC route walk')

status_out = W.Output(layout={'border': '1px solid #ccc', 'padding': '8px'})
plot_out = W.Output(layout={'border': '1px solid #ccc', 'padding': '8px'})

def core():
    if state['core'] is None:
        raise RuntimeError('Load overlay first')
    return state['core']

def input_mask():
    return (1 << int(channels_widget.value)) - 1

def format_bool(value):
    return '1' if int(value) else '0'

def apply_dac_settings():
    cfg = core().configure_dac_tone_bank(
        freq_hz=float(frequency_widget.value) * 1_000_000.0,
        amplitude=int(amplitude_widget.value),
        phase_deg_per_channel=float(phase_widget.value),
        enable_mask=input_mask(),
        dac_sample_rate_hz=float(dac_rate_widget.value) * 1_000_000.0,
    )
    return cfg

def configure_pfb_settings():
    return core().configure_channelizer(
        nchan=4096,
        taps=int(pfb_taps_widget.value),
        chan0=int(pfb_chan0_widget.value),
        chan_count=int(pfb_chan_count_widget.value),
        time_count=int(pfb_time_count_widget.value),
        fft_shift=int(pfb_shift_widget.value),
        enable=True,
    )

def compute_preview_spectrum(preview):
    sample_rate = int(preview['sample_rate_hz'])
    count = int(preview['count'])
    freq_hz = np.arange(count, dtype=np.float64) * (sample_rate / count)
    power = {}
    peaks = {}
    for ch, iq in preview['iq'].items():
        arr = np.asarray(iq, dtype=np.float64)
        complex_samples = arr[:, 0] + 1j * arr[:, 1]
        fft = np.fft.fft(complex_samples)
        pwr = np.abs(fft) ** 2
        peak_bin = int(np.argmax(pwr))
        display_bin = min(peak_bin, count - peak_bin)
        power[ch] = pwr
        peaks[ch] = {
            'peak_bin': peak_bin,
            'display_bin': display_bin,
            'peak_mhz': display_bin * sample_rate / count / 1_000_000.0,
            'peak_power': float(pwr[peak_bin]),
            'phase_deg': float(np.angle(fft[peak_bin], deg=True)),
        }
    return {'freq_hz': freq_hz, 'power': power, 'peaks': peaks, 'sample_rate_hz': sample_rate, 'count': count}

def channel_label(ch):
    return 'physical loopback verified' if ch == 0 else 'digital/control only, not analog-verified'

def render_status(status, tone_cfg, preview, spectrum, elapsed_s):
    with status_out:
        clear_output(wait=True)
        print(f'Project root: {PROJECT_ROOT}')
        print(f'CORE_VERSION=0x{status["core_version"]:08x} expected=0x{EXPECTED_CORE_VERSION:08x}')
        print(
            f'streaming={format_bool(status["streaming"])} sample0={preview["sample0"]} '
            f'input_mask=0x{preview["input_mask"]:02x} preview_count={preview["count"]} '
            f'sample_rate={preview["sample_rate_hz"]} Hz elapsed={elapsed_s:.2f}s'
        )
        print(
            f'UDP_DRY_RUN={format_bool(status["udp_dry_run"])} QSFP_LINK_UP={format_bool(status["qsfp_link_up"])} '
            f'TX_FIFO_LEVEL={status.get("tx_fifo_level_words", 0)} '
            f'TX_FIFO_HIGH_WATER={status.get("tx_fifo_high_water_words", 0)}'
        )
        print(
            f'PFB nchan={status.get("pfb_nchan", 0)} chan0={status.get("pfb_chan0", 0)} '
            f'chan_count={status.get("pfb_chan_count", 0)} time_count={status.get("pfb_time_count", 0)} '
            f'frame={status.get("pfb_frame_count", 0)} peak_chan={status.get("pfb_peak_chan", 0)} '
            f'peak_power={status.get("pfb_peak_power", 0)} config_valid={format_bool(status.get("pfb_config_valid", 0))}'
        )
        print(
            f'DAC freq={tone_cfg["freq_hz"] / 1_000_000.0:.3f} MHz rate_cal={tone_cfg["dac_sample_rate_hz"] / 1_000_000.0:.3f} MHz phase_step=0x{tone_cfg["phase_step"]:08x} '
            f'amplitude={tone_cfg["amplitude"]} phase_step/ch={tone_cfg["phase_deg_per_channel"]:.1f} deg'
        )
        for ch in preview['inputs']:
            pk = spectrum['peaks'].get(ch, {})
            print(
                f'CH{ch}: {channel_label(ch)} | peak={pk.get("peak_mhz", 0.0):.3f} MHz '
                f'bin={pk.get("peak_bin", 0)} phase={pk.get("phase_deg", 0.0):.1f} deg'
            )

def render_plots(preview, spectrum):
    freq_mhz = float(frequency_widget.value)
    with plot_out:
        clear_output(wait=True)
        fig, (ax_scope, ax_spec) = plt.subplots(1, 2, figsize=(14, 4.8))
        sample_rate = int(preview['sample_rate_hz'])
        count = int(preview['count'])
        t_us = np.arange(count, dtype=np.float64) / sample_rate * 1_000_000.0
        max_abs = 1.0
        for iq in preview['iq'].values():
            arr = np.asarray(iq)
            if arr.size:
                max_abs = max(max_abs, float(np.max(np.abs(arr[:, 0]))))
        for ch in preview['inputs']:
            arr = np.asarray(preview['iq'][ch])
            color = COLORS[ch % len(COLORS)]
            ax_scope.plot(t_us, arr[:, 0], color=color, lw=1.1, label=f'CH{ch}')
            if reference_widget.value:
                phase_rad = np.deg2rad(float(phase_widget.value) * ch)
                ref = 0.45 * max_abs * np.sin(2 * np.pi * freq_mhz * t_us + phase_rad)
                ax_scope.plot(t_us, ref, color=color, lw=0.7, alpha=0.35, ls='--')
        ax_scope.set_title('Virtual scope')
        ax_scope.set_xlabel('time (us)')
        ax_scope.set_ylabel('ADC code')
        ax_scope.grid(True, alpha=0.3)
        ax_scope.legend(loc='upper right', fontsize=8)

        half = max(2, count // 2)
        freq_axis_mhz = spectrum['freq_hz'][:half] / 1_000_000.0
        for ch in preview['inputs']:
            pwr = np.asarray(spectrum['power'][ch], dtype=np.float64)[:half]
            pwr_db = 10.0 * np.log10(np.maximum(pwr, 1.0))
            color = COLORS[ch % len(COLORS)]
            pk = spectrum['peaks'][ch]
            ax_spec.plot(freq_axis_mhz, pwr_db, color=color, lw=1.1, label=f'CH{ch} {pk["peak_mhz"]:.2f} MHz')
            ax_spec.axvline(pk['peak_mhz'], color=color, alpha=0.18, lw=1.0)
        ax_spec.axvline(freq_mhz, color='#222222', alpha=0.35, lw=1.0, ls='--')
        ax_spec.set_title('Virtual spectrum')
        ax_spec.set_xlabel('frequency (MHz)')
        ax_spec.set_ylabel('power (dB, arbitrary)')
        ax_spec.grid(True, alpha=0.3)
        ax_spec.legend(loc='upper right', fontsize=8)
        plt.tight_layout()
        display(fig)
        plt.close(fig)

def capture_and_render():
    t0 = time.monotonic()
    tone_cfg = apply_dac_settings()
    preview = core().capture_preview(n=int(samples_widget.value), input_mask=input_mask(), timeout=float(timeout_widget.value))
    spectrum = compute_preview_spectrum(preview)
    status = core().read_status()
    state['last_preview'] = preview
    state['last_spectrum'] = spectrum
    elapsed = time.monotonic() - t0
    render_status(status, tone_cfg, preview, spectrum, elapsed)
    render_plots(preview, spectrum)

async def live_loop():
    state['live'] = True
    while state['live']:
        try:
            capture_and_render()
            state['last_error'] = None
        except Exception as exc:
            state['last_error'] = exc
            with status_out:
                clear_output(wait=True)
                print(f'ERROR: {exc}')
            state['live'] = False
            break
        await asyncio.sleep(float(interval_widget.value))

def on_load(_):
    with status_out:
        clear_output(wait=True)
        state['core'] = T510FEngine(str(BITFILE), download=bool(download_widget.value))
        status = core().read_status()
        print(f'Loaded {BITFILE}')
        print(f'CORE_VERSION=0x{status["core_version"]:08x}')

def on_init(_):
    with status_out:
        clear_output(wait=True)
        core().stop()
        pfb_cfg = configure_pfb_settings()
        status = core().init_lab_rfdc(mask=0x0001, mode='spec', tone_enable=True, wait_seconds=float(timeout_widget.value))
        tone_cfg = apply_dac_settings()
        print(f'Initialized Stage 7: streaming={status["streaming"]} CORE_VERSION=0x{status["core_version"]:08x}')
        print(f'PFB chan0={pfb_cfg["pfb_chan0"]} chan_count={pfb_cfg["pfb_chan_count"]} time_count={pfb_cfg["pfb_time_count"]}')
        print(f'DAC phase_step=0x{tone_cfg["phase_step"]:08x}')

def on_apply(_):
    with status_out:
        clear_output(wait=True)
        tone_cfg = apply_dac_settings()
        print(f'DAC applied: freq={tone_cfg["freq_hz"] / 1_000_000.0:.3f} MHz phase_step=0x{tone_cfg["phase_step"]:08x}')

def on_pfb(_):
    with status_out:
        clear_output(wait=True)
        pfb_cfg = configure_pfb_settings()
        print(f'PFB applied: chan0={pfb_cfg["pfb_chan0"]} chan_count={pfb_cfg["pfb_chan_count"]} time_count={pfb_cfg["pfb_time_count"]}')
        print(f'status=0x{pfb_cfg["pfb_status"]:08x} frame={pfb_cfg["pfb_frame_count"]} peak_chan={pfb_cfg["pfb_peak_chan"]} peak_power={pfb_cfg["pfb_peak_power"]}')

def on_header(_):
    with status_out:
        clear_output(wait=True)
        capture = core().capture_tx_header(timeout=float(timeout_widget.value))
        header = capture['header_dict']
        pfb_status = core().read_channelizer_status()
        print('Captured SPEC/PFB dry-run header:')
        for key in ['version', 'stream_type', 'flags', 'frame_id', 'seq_no', 'chan0', 'chan_count', 'time_count', 'ninput', 'payload_format', 'payload_bytes']:
            print(f'{key}: {header[key]}')
        print(f'PFB frame={pfb_status["pfb_frame_count"]} peak_chan={pfb_status["pfb_peak_chan"]} peak_power={pfb_status["pfb_peak_power"]}')

def configure_tx_routes():
    core().configure_tx_control(force_dry_run=True, cmac_enable=False, frame_builder_enable=True, drop_on_route_miss=True)
    core().configure_network(
        src_ip='10.0.1.1',
        src_mac='02:00:00:00:00:01',
        dgx_a={'ip': '10.0.1.10', 'mac': '02:00:00:00:00:0a', 'port': 4100},
        dgx_b={'ip': '10.0.1.11', 'mac': '02:00:00:00:00:0b', 'port': 4200},
        time_dst={'ip': '10.0.1.16', 'mac': '02:00:00:00:00:10', 'port': 4300},
    )
    core().configure_spec_routes([
        {'id': 0, 'chan0': 0, 'chan_count': 2048, 'endpoint_id': 0},
        {'id': 1, 'chan0': 2048, 'chan_count': 2048, 'endpoint_id': 1},
    ])
    core().configure_time_routes([
        {'id': 0, 'input_mask': input_mask(), 'endpoint_id': 2},
    ])
    return core().read_tx_status()

def on_tx_config(_):
    with status_out:
        clear_output(wait=True)
        status = configure_tx_routes()
        print('TX routes configured: SPEC[0..2047]->10.0.1.10:4100, SPEC[2048..4095]->10.0.1.11:4200, TIME mask->10.0.1.16:4300')
        for key in ['tx_control', 'tx_status', 'udp_dry_run_active', 'qsfp_link_up', 'frame_builder_enabled', 'force_dry_run']:
            print(f'{key}: {status.get(key, 0)}')

def on_tx_frame(_):
    with status_out:
        clear_output(wait=True)
        capture = core().capture_tx_frame_header(timeout=float(timeout_widget.value))
        frame = capture['frame_dict']
        print('Captured Ethernet/IPv4/UDP frame header:')
        for key in ['dst_mac_str', 'src_mac_str', 'src_ip_str', 'dst_ip_str', 'src_port', 'dst_port', 'ipv4_total_length', 'udp_length', 'payload_len']:
            print(f'{key}: {frame[key]}')
        print('Note: frame capture is 128B, so full 128B T510 payload header is verified with Capture PFB header.')

def on_tx_walk(_):
    with status_out:
        clear_output(wait=True)
        configure_tx_routes()
        results = []
        for chan0 in (0, 2048):
            core().configure_channelizer(chan0=chan0, chan_count=int(pfb_chan_count_widget.value), time_count=int(pfb_time_count_widget.value))
            time.sleep(0.05)
            capture = core().capture_tx_frame_header(timeout=float(timeout_widget.value))
            frame = capture['frame_dict']
            results.append((chan0, frame['dst_ip_str'], frame['dst_port'], frame['dst_mac_str']))
        for chan0, ip, port, mac in results:
            print(f'chan0={chan0}: dst={ip}:{port} mac={mac}')

def on_capture(_):
    capture_and_render()

def on_start(_):
    if state['task'] is not None and not state['task'].done():
        return
    state['task'] = asyncio.create_task(live_loop())

def on_stop(_):
    state['live'] = False
    task = state.get('task')
    if task is not None and not task.done():
        task.cancel()
    with status_out:
        print('Live refresh stopped')

def on_status(_):
    with status_out:
        clear_output(wait=True)
        status = core().read_status()
        for key in ['core_version', 'sync_config', 'status', 'streaming', 'rfdc_status_flags', 'rfdc_current_valid_mask', 'preview_status', 'preview_input_mask', 'tx_link_status_flags', 'tx_status', 'tx_control', 'tx_frame_built_count', 'tx_frame_sent_count', 'tx_frame_dropped_count', 'tx_frame_byte_count', 'tx_route_miss_count', 'tx_route_error_count', 'tx_fifo_level_words', 'tx_fifo_high_water_words', 'dac_enable_mask', 'pfb_status', 'pfb_nchan', 'pfb_chan0', 'pfb_chan_count', 'pfb_time_count', 'pfb_frame_count', 'pfb_peak_chan', 'pfb_peak_power']:
            value = status.get(key, 0)
            if key in {'core_version', 'sync_config', 'status', 'rfdc_status_flags', 'preview_status', 'preview_input_mask', 'tx_link_status_flags', 'tx_status', 'tx_control', 'dac_enable_mask', 'pfb_status'}:
                print(f'{key}: 0x{value:08x}')
            else:
                print(f'{key}: {value}')

load_button.on_click(on_load)
init_button.on_click(on_init)
apply_button.on_click(on_apply)
pfb_button.on_click(on_pfb)
header_button.on_click(on_header)
tx_config_button.on_click(on_tx_config)
tx_frame_button.on_click(on_tx_frame)
tx_walk_button.on_click(on_tx_walk)
capture_button.on_click(on_capture)
start_button.on_click(on_start)
stop_button.on_click(on_stop)
status_button.on_click(on_status)

display(W.VBox([
    W.HBox([download_widget, reference_widget]),
    W.HBox([frequency_widget, dac_rate_widget, amplitude_widget, phase_widget]),
    W.HBox([pfb_chan0_widget, pfb_chan_count_widget, pfb_time_count_widget, pfb_taps_widget, pfb_shift_widget]),
    W.HBox([channels_widget, samples_widget, interval_widget, timeout_widget]),
    W.HBox([load_button, init_button, apply_button, pfb_button, header_button, tx_config_button, tx_frame_button, tx_walk_button]),
    W.HBox([capture_button, start_button, stop_button, status_button]),
    status_out,
    plot_out,
]))

print('Stage 7 default: 64 channels x 4 times x 8 inputs x IQ16 = 8192B SPEC payload, wrapped in Ethernet/IPv4/UDP during TX preflight. CH1..CH7 remain digital/control only until cabled.')
